In [3]:
import numpy as np
import pandas as pd
import sklearn as skl
import statsmodels as sm

In [4]:
import quandl
import matplotlib as mp

Generate Data With Heteroskedasticity- Simulating heteroskedasticity intentionally

In [5]:
np.random.seed(42)

In [ ]:
e = np.random.normal(0, np.abs(X), 500)

In [16]:
X = np.random.normal(0, 1, 500)
e = np.random.normal(0, np.abs(X), 500) #Heteroskedastic error- making the variance of errors not constant

import statsmodels.api as sm #import the statsmodels.api submodule
import statsmodels.stats.diagnostic as smd #contains statistical tests

In [12]:
Y = 2 + 1.5*X + e  # regression model

X_const = sm.add_constant(X) # adds a column of 1s to estimate the intercept

model = sm.OLS(Y, X_const).fit() # estimate coefficient using OLS

In [ ]:
#coefficients are still unbiased even with heteroskedasticity
#The violation of Guass Markov assumption here affects the Standard Errors- Unreliable SE

Standard Versus Robust Errors

In [14]:
print("Standard SE:", model.bse) #standard error assuming homoskedasticity

Standard SE: [0.04704741 0.0479931 ]


Since we know errors are heteroskedastic, the SEs above are biased and leads to:
Incorrect t-statistics
Misleading significance

In [15]:
model_hc3 = model.get_robustcov_results(cov_type='HC3') #robust standard error(hc3)

print("Robust HC3 SE:", model_hc3.bse) # Robust SEs is larger — reflecting true uncertainty

Robust HC3 SE: [0.04747773 0.10733555]


Robust SE > SE
Reflects True Variability
Lesson: OLS coefficients can survive heteroskedasticity because the coefficient (Beta) remains unbiased, but the Inference is affected and reflects in wrong standard errors and incorrect T-stats. 

Testing for Heteroskedasticity, Incorporating the 2-solution Method to Heteroskedasticity

In [17]:
#simulate sectional wage data
np.random.seed(0)
n = 600
education = np.random.uniform(8, 20, n)
experience = np.random.uniform(0, 40, n)

# Variance grows with education 

wage = 15000 + 2000*education + 500*experience + np.random.normal(0, 1500*education, n)

#Constructing regression matrix
X = sm.add_constant(np.column_stack([education, experience]))

#Run OLS regression

ols = sm.OLS(wage, X).fit()

In [18]:
#Breusch-Pagan Test
bp_test = smd.het_breuschpagan(ols.resid, X)  #tests for heteroskedasticity by plotting the residuals with the regressors

# Null hypothesis = Homoskedasticty
  # - Alternative hypothesis = Heteroskedasticity
    # -- Reject Null hypothesis if P-value < 0.05

print(f"Breusch-Pagan LM stat: {bp_test[0]:.3f}")
print(f"p-value: {bp_test[1]:.4f}")

Breusch-Pagan LM stat: 38.526
p-value: 0.0000


In [19]:
#Recompute Robust Standard Errors using H3 --- Solution 1 to heteroskedasticity
ols_robust = ols.get_robustcov_results(cov_type='HC3')

#Compare standard Vs Robust SE

print(" Coeff | OLS SE | HC3 SE")
for i, name in enumerate(['const','education','experience']):
    print(f"{name:12}: {ols.bse[i]:8.1f} | {ols_robust.bse[i]:8.1f}")



 Coeff | OLS SE | HC3 SE
const       :   3614.8 |   3357.9
education   :    236.2 |    238.0
experience  :     71.2 |     72.9


In [20]:
# -- Solution 2 to Heteroskedasticity -- Weighter Least Square (WLS)
#  Model variance structure 
# Get squared residuals (proxy for variance)

resid_sq = ols.resid**2

# Regress squared residuals on X
# This estimates: Var(e_i) = f(X_i)
var_model = sm.OLS(resid_sq, X).fit()

# Predicted variance for each observation
predicted_var = var_model.fittedvalues

# Avoid division by zero
predicted_var = np.clip(predicted_var, 1e-6, None)

# Weights = inverse of variance
weights = 1 / predicted_var

# WLS estimation 
wls = sm.WLS(wage, X, weights=weights).fit()

print("\nCoefficient Comparison:")
print("           OLS       WLS")
for i, name in enumerate(['const','education','experience']):
    print(f"{name:10}: {ols.params[i]:8.2f}  {wls.params[i]:8.2f}")


Coefficient Comparison:
           OLS       WLS
const     : 15573.15  14854.36
education :  2138.56   2169.66
experience:   434.64    449.02
